# 🤟 Wasel v4 Pro — Local AI (No API Key!)
### SmolVLM Vision Model on Colab GPU

1. Cell 1: Install + Download model (~2 min)
2. Cell 2: Launch — no API key, no internet needed!

> **البديل المحلي — الموديل يعمل على GPU مباشرة بدون مفتاح!**

In [ ]:
#@title 📦 Cell 1: Install & Download Model

!pip install -q transformers accelerate flask flask-cors pillow torch

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

print("\n⏳ Downloading vision model...")
from transformers import AutoProcessor, AutoModelForVision2Seq

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
device = next(model.parameters()).device
print(f"✅ Model on {device} | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")
print("🎉 Run Cell 2!")

In [ ]:
#@title 🎥 Cell 2: Launch Live Translator

import threading, time, base64, io
from flask import Flask, request, jsonify, Response
from flask_cors import CORS
from PIL import Image

PROMPT = """Look at this image. Is the person making a sign language gesture?
If yes: reply ONLY the meaning in 1-3 words.
If no: reply ...
Reply ONLY the word."""

def ask_ai(pil_image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": pil_image},
        {"type": "text", "text": PROMPT}
    ]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=[pil_image], return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    result = processor.decode(new_tokens, skip_special_tokens=True).strip()
    return result if result else "..."

print("⏳ Warming up...")
_ = ask_ai(Image.new("RGB", (100, 100)))
print("✅ Ready!")

app = Flask(__name__)
CORS(app)

PAGE = r"""
<!DOCTYPE html><html><head><meta charset='utf-8'>
<meta name='viewport' content='width=device-width,initial-scale=1'>
<title>Wasel v4 Pro</title>
<style>
*{margin:0;padding:0;box-sizing:border-box}
body{background:#0a0a0a;font-family:'Segoe UI',sans-serif;overflow:hidden;height:100vh}
#c{position:relative;width:100vw;height:100vh}
video{width:100%;height:100%;object-fit:cover;transform:scaleX(-1)}
#top{position:absolute;top:0;left:0;right:0;padding:18px 28px;
background:linear-gradient(180deg,rgba(0,0,0,0.85) 0%,transparent 100%)}
#brand{color:#666;font-size:13px;letter-spacing:3px;text-transform:uppercase}
#txt{color:#00ff88;font-size:42px;font-weight:700;margin-top:6px;
text-shadow:0 2px 20px rgba(0,255,136,0.4);min-height:55px;transition:all .3s}
#bot{position:absolute;bottom:16px;right:24px;color:#444;font-size:12px}
</style></head><body>
<div id='c'>
<video id='v' autoplay playsinline muted></video>
<div id='top'><div id='brand'>WASEL v4 PRO — LOCAL AI (ON-DEVICE)</div>
<div id='txt'>Starting camera...</div></div>
<div id='bot'>SmolVLM — Local GPU</div>
</div>
<canvas id='cv' style='display:none'></canvas>
<script>
const v=document.getElementById('v'),cv=document.getElementById('cv'),
cx=cv.getContext('2d'),tx=document.getElementById('txt'),
bt=document.getElementById('bot');
let busy=false;
navigator.mediaDevices.getUserMedia({video:{width:640,height:480,facingMode:'user'}})
.then(s=>{v.srcObject=s;tx.textContent='Show a sign...';tx.style.color='#555';
setInterval(go,3000)}).catch(e=>{tx.textContent='Camera: '+e.message});
function go(){if(busy)return;busy=true;cv.width=384;cv.height=288;
cx.drawImage(v,0,0,384,288);
const d=cv.toDataURL('image/jpeg',0.5);
bt.textContent='🧠 Thinking...';
fetch('/translate',{method:'POST',headers:{'Content-Type':'application/json'},
body:JSON.stringify({image:d})})
.then(r=>r.json()).then(d=>{
const t=d.translation||'...';
if(t==='...'||t.length<2){tx.textContent='Show a sign...';tx.style.color='#555'}
else{tx.textContent=t;tx.style.color='#00ff88'}
bt.textContent='🧠 Local — '+d.ms+'ms — '+new Date().toLocaleTimeString();busy=false
}).catch(e=>{bt.textContent='Err: '+e;busy=false})}
</script></body></html>
"""

@app.route('/')
def index():
    return Response(PAGE, mimetype='text/html')

@app.route('/translate', methods=['POST'])
def translate():
    t0 = time.time()
    try:
        d = request.json
        img_b = base64.b64decode(d['image'].split(',')[1])
        pil = Image.open(io.BytesIO(img_b)).convert('RGB')
        result = ask_ai(pil)
        ms = int((time.time()-t0)*1000)
        print(f"'{result}' ({ms}ms)")
        return jsonify({'translation': result, 'ms': ms})
    except Exception as e:
        return jsonify({'translation': f'Error: {str(e)[:40]}', 'ms': 0})

threading.Thread(target=lambda: app.run(port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(2)

from google.colab import output
output.serve_kernel_port_as_window(5000)

print("\n══════════════════════════════════════")
print("  🤟 Wasel v4 Pro — LOCAL AI")
print("══════════════════════════════════════")
print("  🧠 SmolVLM on GPU — no API key!")
print("  📸 Allow camera → show a sign!")
print("══════════════════════════════════════")